# 🐐 Goat Skin Disease Detector — Dedicated Model

Part of your SIH multi-species project: this notebook is **goat-only** (no cow, no sheep mixed in), so the CNN
never has to reconcile different photo framings across species — this is the same fix that solved the cow
model's calibration problem. Trains on **`goat_dataset.zip`** (5 diseases, bounding-box crops) plus
**`HealthyGoat_dataset.zip`** (whole-image Healthy class), and gives you a `.keras` file + a symptom `.joblib`
file, ready to drop into your website's backend alongside the cow and sheep models.

**Final unified classes for this model (6):**
1. `Mange`
2. `Caseous_Lymphadenitis` (Cheesy Gland)
3. `Ringworm`
4. `Contagious_Ecthyma_Orf`
5. `Lice_Infestation`
6. `Normal_Healthy_Skin`

**Output format matches the cow notebook exactly:** one direct final verdict (condition + confidence, via
Test-Time Augmentation), no percentage breakdown table for the photo mode; the symptom mode keeps its full
breakdown, same as before.


## 0. GPU check

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("GPU found:", gpus if gpus else "⚠️ NO GPU — go to Runtime → Change runtime type → GPU (T4) and restart")


## 1. Install & import everything

In [ ]:
!pip install -q gradio scikit-learn seaborn

import os, glob, shutil, json, random, zipfile, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


## 2. Load `goat_dataset.zip` and `HealthyGoat_dataset.zip` from Google Drive

Set `DRIVE_FOLDER` if your zips aren't directly in **My Drive**. Filename matching is case-insensitive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# <<< EDIT THIS if your zips are inside a subfolder of My Drive >>>
DRIVE_FOLDER = '/content/drive/MyDrive'

RAW_DIR = '/content/data/raw_goat'
os.makedirs(RAW_DIR, exist_ok=True)

EXPECTED_NAMES = {
    'goat':        'goat_dataset.zip',
    'healthygoat': 'HealthyGoat_dataset.zip',
}

def find_case_insensitive(folder, filename):
    target = filename.lower()
    try:
        for f in os.listdir(folder):
            if f.lower() == target:
                return os.path.join(folder, f)
    except FileNotFoundError:
        pass
    return os.path.join(folder, filename)

def check_zip(path, min_kb=10):
    if not os.path.exists(path):
        return False, "not found in that Drive folder"
    size_kb = os.path.getsize(path) / 1024
    if size_kb < min_kb:
        return False, f"only {size_kb:.0f} KB — likely an incomplete upload to Drive"
    if not zipfile.is_zipfile(path):
        return False, f"{size_kb:.0f} KB but not a valid zip file (corrupted)"
    return True, f"OK ({size_kb:.0f} KB)"

zip_paths = {}
print("Checking Drive for expected files:")
for key, fname in EXPECTED_NAMES.items():
    path = find_case_insensitive(DRIVE_FOLDER, fname)
    ok, msg = check_zip(path)
    print(f"  {key:14s} {path:60s} -> {msg}")
    if ok:
        zip_paths[key] = path

missing = set(EXPECTED_NAMES) - set(zip_paths)
assert not missing, f"Missing/invalid in Drive: {missing}. Check DRIVE_FOLDER and filenames."

for key, zp in zip_paths.items():
    out_dir = os.path.join(RAW_DIR, key)
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zp, 'r') as z:
        z.extractall(out_dir)
    print(f"Extracted {key} -> {out_dir}")


## 3. Auto-locate the annotation CSVs and the Healthy Goat image folder

In [ ]:
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')

goat_csvs = glob.glob(os.path.join(RAW_DIR, 'goat', '**', '_annotations.csv'), recursive=True)
print("Goat annotation files found:")
for c in goat_csvs:
    print(" ", c)

healthygoat_root = RAW_DIR + '/healthygoat'
all_hg_dirs = glob.glob(os.path.join(healthygoat_root, '**/'), recursive=True) + [healthygoat_root + '/']
healthygoat_dir = max(all_hg_dirs, key=lambda d: len([f for f in glob.glob(os.path.join(d, '*')) if f.lower().endswith(IMG_EXTS)]))
n_hg = len([f for f in glob.glob(os.path.join(healthygoat_dir, '*')) if f.lower().endswith(IMG_EXTS)])
print(f"\nHealthy Goat dir: {healthygoat_dir} -> {n_hg} files")


## 4. Build the unified image dataset

Disease images are bounding-box exports → every lesion box is **cropped out** as its own training image (removes
background clutter). Healthy images are plain whole-image files → copied directly (re-encoded as clean RGB JPEG,
handles any stray GIFs safely).

In [ ]:
UNIFIED_DIR = '/content/data/unified_goat'
if os.path.exists(UNIFIED_DIR):
    shutil.rmtree(UNIFIED_DIR)

CLASS_MAP = {
    'Mange':             'Mange',
    'Cheesy gland':      'Caseous_Lymphadenitis',
    'Ringworm':          'Ringworm',
    'Contagious ecthyma':'Contagious_Ecthyma_Orf',
    'Lice infestation':  'Lice_Infestation',
}

def ensure_dir(d):
    os.makedirs(d, exist_ok=True)
    return d

def copy_whole_images(src_dir, unified_class, prefix='img'):
    out_dir = ensure_dir(os.path.join(UNIFIED_DIR, unified_class))
    n = 0
    for f in glob.glob(os.path.join(src_dir, '*')):
        if not f.lower().endswith(IMG_EXTS + ('.gif',)):
            continue
        try:
            img = Image.open(f)
            if getattr(img, 'is_animated', False):
                img.seek(0)
            img = img.convert('RGB')
            img.save(os.path.join(out_dir, f"{prefix}_{n:05d}.jpg"), quality=95)
            n += 1
        except Exception:
            continue
    return n

def crop_bboxes_from_csv(csv_path, prefix, pad_frac=0.08):
    img_dir = os.path.dirname(csv_path)
    df = pd.read_csv(csv_path)
    counts = {}
    for i, row in df.iterrows():
        cls = str(row['class']).strip()
        if cls not in CLASS_MAP:
            continue
        unified_class = CLASS_MAP[cls]
        img_path = os.path.join(img_dir, row['filename'])
        if not os.path.exists(img_path):
            continue
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            continue
        w, h = img.size
        xmin, ymin, xmax, ymax = row['xmin'], row['ymin'], row['xmax'], row['ymax']
        bw, bh = xmax - xmin, ymax - ymin
        pad_x, pad_y = bw * pad_frac, bh * pad_frac
        xmin = max(0, int(xmin - pad_x)); ymin = max(0, int(ymin - pad_y))
        xmax = min(w, int(xmax + pad_x)); ymax = min(h, int(ymax + pad_y))
        if xmax <= xmin or ymax <= ymin:
            continue
        crop = img.crop((xmin, ymin, xmax, ymax))
        out_dir = ensure_dir(os.path.join(UNIFIED_DIR, unified_class))
        counts[unified_class] = counts.get(unified_class, 0) + 1
        crop.save(os.path.join(out_dir, f"{prefix}_{i:05d}.jpg"), quality=95)
    return counts

goat_totals = {}
for csv_path in goat_csvs:
    counts = crop_bboxes_from_csv(csv_path, prefix='goat')
    for k, v in counts.items():
        goat_totals[k] = goat_totals.get(k, 0) + v
print("Disease crops:", goat_totals)

n_healthy = copy_whole_images(healthygoat_dir, 'Normal_Healthy_Skin', prefix='healthy')
print(f"Normal_Healthy_Skin: {n_healthy} images")

print("\n=== FINAL CLASS COUNTS ===")
class_names_unified = sorted(os.listdir(UNIFIED_DIR))
for c in class_names_unified:
    n = len(glob.glob(os.path.join(UNIFIED_DIR, c, '*')))
    print(f"  {c:25s} {n:5d} images")


In [ ]:
counts = {c: len(glob.glob(os.path.join(UNIFIED_DIR, c, '*'))) for c in class_names_unified}
plt.figure(figsize=(9, 4))
sns.barplot(x=list(counts.values()), y=list(counts.keys()), palette='viridis')
plt.xlabel('Number of images')
plt.title('Goat dataset — class balance')
plt.tight_layout()
plt.show()


## 5. Sanity-check a few crops per class

In [ ]:
fig, axes = plt.subplots(len(class_names_unified), 4, figsize=(12, 3 * len(class_names_unified)))
for row, c in enumerate(class_names_unified):
    files_c = glob.glob(os.path.join(UNIFIED_DIR, c, '*'))
    sample = random.sample(files_c, min(4, len(files_c)))
    for col in range(4):
        ax = axes[row, col]
        ax.axis('off')
        if col < len(sample):
            ax.imshow(Image.open(sample[col]))
        if col == 0:
            ax.set_title(c, loc='left', fontsize=9)
plt.tight_layout()
plt.show()


## 6. Stratified train / val / test split (70 / 15 / 15)

In [ ]:
SPLIT_ROOT = '/content/data/split_goat'
if os.path.exists(SPLIT_ROOT):
    shutil.rmtree(SPLIT_ROOT)

def list_images(d):
    return [f for f in glob.glob(os.path.join(d, '*')) if f.lower().endswith(IMG_EXTS)]

for c in class_names_unified:
    files_c = list_images(os.path.join(UNIFIED_DIR, c))
    train, temp = train_test_split(files_c, test_size=0.30, random_state=SEED)
    val, test = train_test_split(temp, test_size=0.50, random_state=SEED)
    for split_name, split_files in [('train', train), ('val', val), ('test', test)]:
        out_dir = os.path.join(SPLIT_ROOT, split_name, c)
        os.makedirs(out_dir, exist_ok=True)
        for f in split_files:
            shutil.copy(f, os.path.join(out_dir, os.path.basename(f)))
    print(f"{c:25s} train={len(train):4d}  val={len(val):4d}  test={len(test):4d}")


## 7. Build `tf.data` datasets — 260×260, categorical (6-way)

In [ ]:
IMG_SIZE = (260, 260)
BATCH_SIZE = 32

def make_dataset(split_name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        os.path.join(SPLIT_ROOT, split_name),
        image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        label_mode='categorical', shuffle=shuffle, seed=SEED,
    )

train_ds = make_dataset('train', shuffle=True)
val_ds   = make_dataset('val',   shuffle=False)
test_ds  = make_dataset('test',  shuffle=False)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Classes (index order):", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)


## 8. Class weights (Mange has far more images than Lice Infestation)

In [ ]:
train_counts = {c: len(glob.glob(os.path.join(SPLIT_ROOT, 'train', c, '*'))) for c in class_names}
y_train_flat = []
for i, c in enumerate(class_names):
    y_train_flat += [i] * train_counts[c]
y_train_flat = np.array(y_train_flat)

class_weight_arr = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train_flat)
class_weight = {i: w for i, w in enumerate(class_weight_arr)}
for i, c in enumerate(class_names):
    print(f"  {c:25s} weight={class_weight[i]:.3f}  (n={train_counts[c]})")


## 9. Build the model — EfficientNetB0, 6-way softmax

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
], name='augmentation')

base_model = tf.keras.applications.EfficientNetB0(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet',
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.4)(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name='goat_skin_disease_classifier')
model.summary()


## 10. Train — Phase A: head only (backbone frozen)

In [ ]:
checkpoint_path = 'goat_skin_best.keras'

callbacks_head = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True),
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy'],
)

history_head = model.fit(
    train_ds, validation_data=val_ds, epochs=15,
    class_weight=class_weight, callbacks=callbacks_head,
)


## 11. Train — Phase B: deep fine-tune (70% of backbone unfrozen)

In [ ]:
base_model.trainable = True
n_layers = len(base_model.layers)
for layer in base_model.layers[: int(n_layers * 0.30)]:
    layer.trainable = False

callbacks_finetune = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy'],
)

history_finetune = model.fit(
    train_ds, validation_data=val_ds, epochs=30,
    class_weight=class_weight, callbacks=callbacks_finetune,
)


In [ ]:
def plot_history(h1, h2, metric):
    v1, v2 = h1.history[metric], h2.history[metric]
    vv1, vv2 = h1.history[f'val_{metric}'], h2.history[f'val_{metric}']
    plt.figure(figsize=(9, 4))
    plt.plot(v1 + v2, label='train')
    plt.plot(vv1 + vv2, label='val')
    plt.axvline(len(v1) - 0.5, color='gray', linestyle='--', label='fine-tune starts')
    plt.title(metric); plt.xlabel('epoch'); plt.legend()
    plt.show()

plot_history(history_head, history_finetune, 'accuracy')
plot_history(history_head, history_finetune, 'loss')


## 12. Load the best checkpoint

In [ ]:
model = tf.keras.models.load_model(checkpoint_path)
print(f"Loaded best checkpoint from {checkpoint_path}")


## 13. Evaluate on the held-out TEST set

In [ ]:
y_true, y_pred_prob = [], []
for images, labels in test_ds:
    y_true.extend(np.argmax(labels.numpy(), axis=1).tolist())
    y_pred_prob.extend(model.predict(images, verbose=0).tolist())

y_true = np.array(y_true)
y_pred_prob = np.array(y_pred_prob)
y_pred = np.argmax(y_pred_prob, axis=1)

test_accuracy = accuracy_score(y_true, y_pred)
print(f"TEST ACCURACY: {test_accuracy:.1%}\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

if test_accuracy >= 0.92:
    print("✅ Target of >92% accuracy reached.")
else:
    print("⚠️ Below 92% — see the troubleshooting section at the end of this notebook.")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'Confusion matrix — test accuracy {test_accuracy:.1%}')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 14. Save the model and download it

In [ ]:
model.save('goat_skin_disease_model.keras')
with open('goat_class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

from google.colab import files
files.download('goat_skin_disease_model.keras')
files.download('goat_class_names.json')


## 15. Symptom-based predictor for Goat (no image needed)

Veterinary knowledge base + synthetic dataset, same approach as the cow and multi-species notebooks — **not** a
real labelled symptom dataset (none of your zips contain one).

In [ ]:
SYMPTOM_KB = {
    'Mange': {
        'core': [
            'intense itching / scratching', 'thickened crusty skin', 'hair loss patches',
            'skin thickening and wrinkling', 'lesions start on head ears and legs',
        ],
        'support': [
            'restlessness', 'weight loss', 'secondary skin infection', 'visible mites under skin scraping',
        ],
    },
    'Caseous_Lymphadenitis': {
        'core': [
            'firm abscess under skin', 'swollen lymph nodes', 'thick cheesy/pus-like discharge if abscess ruptures',
            'abscess most common near jaw neck or shoulder',
        ],
        'support': [
            'weight loss', 'chronic recurring abscesses', 'reduced productivity', 'mild fever',
        ],
    },
    'Ringworm': {
        'core': [
            'circular crusty grey-white patches', 'hair loss in circular pattern',
            'lesions mainly on head neck and ears', 'mild to no itching',
        ],
        'support': [
            'scaly flaky skin', 'lesions spread slowly outward', 'contagious to other animals and humans',
        ],
    },
    'Contagious_Ecthyma_Orf': {
        'core': [
            'scabby pustular lesions around mouth and lips', 'lesions on nose and muzzle',
            'thick crusty scabs', 'lesions on lips can make eating painful',
        ],
        'support': [
            'lesions sometimes on udder or feet', 'young animals affected most severely',
            'lameness if lesions on feet', 'reduced feeding due to mouth pain', 'mild fever',
        ],
    },
    'Lice_Infestation': {
        'core': [
            'visible lice or nits in coat', 'excessive scratching and rubbing',
            'rough dull coat', 'hair loss from rubbing against objects',
        ],
        'support': [
            'restlessness', 'anemia in heavy infestation', 'weight loss', 'skin irritation and redness',
        ],
    },
    'Normal_Healthy_Skin': {
        'core': [
            'smooth uniform coat', 'no nodules or lesions', 'normal appetite',
            'normal body temperature', 'alert and active',
        ],
        'support': [
            'shiny healthy coat', 'no discharge from eyes or nose', 'no swelling or abscesses',
        ],
    },
}

ALL_SYMPTOMS = sorted({s for d in SYMPTOM_KB.values() for s in d['core'] + d['support']})
print(f"Total distinct symptoms: {len(ALL_SYMPTOMS)}")
for disease, info in SYMPTOM_KB.items():
    print(f"\n{disease}")
    print("  core   :", ', '.join(info['core']))
    print("  support:", ', '.join(info['support']))


## 16. Generate a synthetic symptom dataset and train the Random Forest

In [ ]:
def sample_symptom_vector(disease, rng):
    info = SYMPTOM_KB[disease]
    vec = {s: 0 for s in ALL_SYMPTOMS}
    for s in info['core']:
        vec[s] = int(rng.random() < 0.85)
    for s in info['support']:
        vec[s] = int(rng.random() < 0.45)
    other_symptoms = [s for s in ALL_SYMPTOMS if s not in info['core'] and s not in info['support']]
    n_noise = rng.choice([0, 0, 0, 1, 1, 2])
    for s in rng.choice(other_symptoms, size=min(n_noise, len(other_symptoms)), replace=False):
        vec[s] = 1
    return vec

rng = np.random.default_rng(SEED)
N_PER_CLASS = 400
rows = []
for disease in SYMPTOM_KB:
    for _ in range(N_PER_CLASS):
        vec = sample_symptom_vector(disease, rng)
        vec['disease'] = disease
        rows.append(vec)

symptom_df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)

X = symptom_df[ALL_SYMPTOMS].values
y = symptom_df['disease'].values
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

symptom_model = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=2, random_state=SEED, class_weight='balanced', n_jobs=-1,
)
symptom_model.fit(X_train_s, y_train_s)
symptom_acc = accuracy_score(y_test_s, symptom_model.predict(X_test_s))
print(f"Symptom-model held-out accuracy (on synthetic test data): {symptom_acc:.1%}")
print(classification_report(y_test_s, symptom_model.predict(X_test_s), digits=3))

joblib.dump(symptom_model, 'goat_symptom_model.joblib')
with open('goat_symptom_list.json', 'w') as f:
    json.dump(ALL_SYMPTOMS, f, indent=2)
files.download('goat_symptom_model.joblib')
files.download('goat_symptom_list.json')

def predict_from_symptoms(selected_symptoms):
    vec = np.array([[1 if s in selected_symptoms else 0 for s in ALL_SYMPTOMS]])
    probs = symptom_model.predict_proba(vec)[0]
    classes = symptom_model.classes_
    order = np.argsort(probs)[::-1]
    return [(classes[i], float(probs[i])) for i in order]

print("\nTest:", predict_from_symptoms(SYMPTOM_KB['Mange']['core']))


## 17. Test-Time Augmentation (TTA) for image predictions

Averages the model's prediction over 8 augmented views (flip, zoom, brightness) of the uploaded photo, so a
correctly-identified condition comes back with high, stable confidence instead of a single noisy pass.

In [ ]:
def _tta_views(pil_img):
    img = pil_img.convert('RGB').resize(IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    views = [arr, tf.image.flip_left_right(arr).numpy()]
    h, w = IMG_SIZE
    for zoom in (0.90, 0.95):
        ch, cw = int(h * zoom), int(w * zoom)
        top, left = (h - ch) // 2, (w - cw) // 2
        cropped = arr[top:top + ch, left:left + cw, :]
        resized = tf.image.resize(cropped, IMG_SIZE).numpy()
        views.append(resized)
        views.append(tf.image.flip_left_right(resized).numpy())
    for delta in (-0.15, 0.15):
        views.append(tf.image.adjust_brightness(arr, delta).numpy())
    return np.stack(views, axis=0)

def predict_from_image_tta(pil_img):
    batch = _tta_views(pil_img)
    probs = model.predict(batch, verbose=0)
    mean_probs = probs.mean(axis=0)
    top_idx = int(np.argmax(mean_probs))
    return class_names[top_idx], float(mean_probs[top_idx])

test_img_path = glob.glob(os.path.join(SPLIT_ROOT, 'test', class_names[0], '*'))[0]
print("Image-mode test (TTA final verdict):", predict_from_image_tta(Image.open(test_img_path)))


## 18. 🚀 Gradio dashboard — Photo mode (TTA-backed) + Symptom mode

Same format as the cow notebook: photo mode gives one direct final verdict, no breakdown table; symptom mode
gives its own verdict plus a full breakdown for anyone who wants it.

In [ ]:
import gradio as gr

def verdict_markdown(label, confidence):
    pretty = label.replace('_', ' ')
    is_healthy = 'Normal' in label or 'Healthy' in label
    icon = '✅' if is_healthy else '🔴'
    return f"## {icon} Final Diagnosis: **{pretty}**\n\n**Confidence: {confidence:.1%}**"

def gr_predict_image(img):
    if img is None:
        return "Please upload a photo of the goat's skin."
    label, confidence = predict_from_image_tta(img)
    return verdict_markdown(label, confidence)

def gr_predict_symptoms(selected):
    if not selected:
        return "Please select at least one symptom.", None
    results = predict_from_symptoms(selected)
    top_label, top_prob = results[0]
    verdict = verdict_markdown(top_label, top_prob)
    breakdown = {label.replace('_', ' '): prob for label, prob in results}
    return verdict, breakdown

with gr.Blocks(title="Goat Skin Disease Detector") as demo:
    gr.Markdown("# 🐐 Goat Skin Disease Detector")
    gr.Markdown(
        "Two independent modes — use whichever fits what you have on hand. "
        "This is a decision-support tool, **not** a replacement for a veterinarian."
    )

    with gr.Tab("📷 Diagnose from Photo"):
        with gr.Row():
            with gr.Column():
                img_input = gr.Image(type="pil", label="Upload a skin photo")
                img_btn = gr.Button("Diagnose", variant="primary")
            with gr.Column():
                img_output = gr.Markdown()
        img_btn.click(gr_predict_image, inputs=img_input, outputs=img_output)

    with gr.Tab("📋 Diagnose from Symptoms"):
        gr.Markdown("Tick every symptom you currently observe, then click Diagnose.")
        sym_checkbox = gr.CheckboxGroup(ALL_SYMPTOMS, label="Symptoms")
        sym_btn = gr.Button("Diagnose", variant="primary")
        sym_verdict = gr.Markdown()
        with gr.Accordion("See full breakdown", open=False):
            sym_breakdown = gr.Label(num_top_classes=len(SYMPTOM_KB), label=None)
        sym_btn.click(gr_predict_symptoms, inputs=sym_checkbox, outputs=[sym_verdict, sym_breakdown])

demo.launch(share=True, debug=False)


## 19. If test accuracy is still below 92%

1. **`Lice_Infestation` has the fewest images (~100-150 crops)** — its recall will be the shakiest of the 6
   classes; gathering more lice images helps more than any model tweak.
2. **Loosen/tighten `pad_frac`** in Section 4 if crops are cutting off too much or too little context around the
   lesion.
3. **Try EfficientNetV2S** as a drop-in replacement for EfficientNetB0 in Section 9.
4. **Check for near-duplicate images across train/test** with `imagehash` — some Roboflow exports contain
   augmented copies of the same base photo, which can inflate the test number if a near-duplicate leaks across
   the split.
